# Map units, fit shared parameters, export an honest report

**SYNTHETIC / UNVERIFIED / DEMO only.** MIT licence (repository LICENSE); generator seed 619. No external dataset fetched/digitised. Conditional profiles and synthetic held-out checks are engineering verification, not independent biological validation.

Install `pip install -e ".[tutorials,reports]"`. Execute from this directory or use `python tutorials/execute_tutorials.py`. Outputs go to tutorials/output (or output when run from tutorials).

In [1]:
import hashlib
import json
from pathlib import Path

from alga_twin.calibration import fit_joint, leave_one_run_out
from alga_twin.calibration_io import load_job, synthetic_job
from alga_twin.fit_report import plain_language, report_html, report_payload, report_pdf
from alga_twin.wizard_data import map_observations, read_wide_csv

config, observations = synthetic_job()
wide = observations.pivot(
    index=["run_id", "time_day", "replicate"], columns="observable", values="value"
).reset_index()
raw = wide.to_csv(index=False).encode()
wide.head()

Out[1]: 
observable       run_id  time_day    replicate         N         X
0           synthetic-0  0.000000  technical-1  8.632220  0.060732
1           synthetic-0  0.714286  technical-1  8.006165  0.065163
2           synthetic-0  1.428571  technical-1  6.238809  0.087648
3           synthetic-0  2.142857  technical-1  5.246674  0.096101
4           synthetic-0  2.857143  technical-1  3.242866  0.105555


Map each column explicitly. Time is elapsed from inoculation and is not shifted. X is dry weight, N is elemental nitrogen (not nitrate-ion mass). Technical repeats have distinct measurement ids; independent biological cultures have distinct run ids. Missing measurement cells are omitted; malformed values refuse.

In [2]:
mapped = map_observations(
    read_wide_csv(raw),
    {"time": "time_day", "X": "X", "N": "N", "run": "run_id", "replicate": "replicate"},
    {"time": "day", "X": "g_DW/liter", "N": "mg_N/liter"},
)
runs, search, errors = load_job(config, mapped)
print("Explicit search assumptions:", config["search"])
print("Fixed noise SD assumptions:", config["error_model"])
print("All biology UNVERIFIED:", all(p.status == "UNVERIFIED" for r in runs for p in r.model.params.values()))

Explicit search assumptions: {'mu_max_day': {'low': 0.3, 'high': 1.8, 'initial': 0.7, 'scale': 'log'}}
Fixed noise SD assumptions: {'kind': 'additive', 'scales': {'X': (0.003, 'g_DW/liter'), 'N': (0.3, 'mg_N/liter')}}
All biology UNVERIFIED: True


In [3]:
result = fit_joint(runs, search, errors)
result.table()

Out[3]: 
    parameter  ...                                             method
0  mu_max_day  ...  Reoptimized profile likelihood; marginal chi-s...

[1 rows x 7 columns]


The profile gate withholds point estimates and intervals for flat/open/failed profiles. Named intervals are marginal asymptotic likelihood intervals, conditional on fixed parameters, initial states and supplied SDs. They have no established empirical coverage. Select a simpler fitting problem or collect informative data if the gate refuses.

In [4]:
for message in plain_language(result):
    print(message)
result.metrics

The fitted curves describe these uploaded measurements. Independent biological validation is absent.
Intervals are conditional on your fixed parameters, initial conditions and specified measurement noise.
The selected parameters passed the numerical profile gate within your search bounds.
SYNTHETIC teaching data: numerical behavior only, not laboratory evidence.
Out[4]: 
        run_id observable      split  ...      rmse       mae        nll
0  synthetic-0          N  in-sample  ...  0.291535  0.233748   1.497190
1  synthetic-0          X  in-sample  ...  0.003716  0.002868 -32.984913
2  synthetic-1          N  in-sample  ...  0.332635  0.283720   2.637331
3  synthetic-1          X  in-sample  ...  0.002141  0.001905 -37.084478
4  synthetic-2          N  in-sample  ...  0.277097  0.215505   1.132285
5  synthetic-2          X  in-sample  ...  0.002251  0.001957 -36.869569

[6 rows x 9 columns]


In [5]:
cv = leave_one_run_out(runs, search, errors)
print(cv.method)
print(cv.folds.to_string(index=False))
cv.metrics

Leave-one-whole-run-out; refit and reprofile training runs only
   held_out status                                            reason
synthetic-0 SCORED Training profiles supported all fitted parameters
synthetic-1 SCORED Training profiles supported all fitted parameters
synthetic-2 SCORED Training profiles supported all fitted parameters
Out[5]: 
         run_id observable      split  ...       mae        nll held_out_fold
0   synthetic-0          N   held-out  ...  0.233776   1.498414   synthetic-0
1   synthetic-0          X   held-out  ...  0.002867 -32.985801   synthetic-0
2   synthetic-1          N  in-sample  ...  0.283470   2.633209   synthetic-0
3   synthetic-1          X  in-sample  ...  0.001907 -37.081392   synthetic-0
4   synthetic-2          N  in-sample  ...  0.214599   1.121801   synthetic-0
5   synthetic-2          X  in-sample  ...  0.001962 -36.858215   synthetic-0
6   synthetic-1          N   held-out  ...  0.284560   2.651243   synthetic-1
7   synthetic-1          X 

Each held-out culture is excluded from fitting/profiling. Training and held-out rows retain separate split labels and units. These are synthetic holdouts, not a biological validation dataset. No pooled score is constructed from only successful folds.

In [6]:
out = Path("output") if Path.cwd().name == "tutorials" else Path("tutorials/output")
out.mkdir(parents=True, exist_ok=True)
(out / "synthetic_observations_wide.csv").write_bytes(raw)
(out / "synthetic_observations_long.csv").write_text(mapped.to_csv(index=False), encoding="utf-8")
(out / "synthetic_job.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
payload = report_payload(
    result,
    title="SYNTHETIC worked example - not biological validation",
    input_sha256=hashlib.sha256(raw).hexdigest(),
    cv=cv,
)
(out / "synthetic_report.html").write_text(report_html(payload), encoding="utf-8")
(out / "synthetic_report.pdf").write_bytes(report_pdf(payload))
(out / "synthetic_report.json").write_text(json.dumps(payload, indent=2, allow_nan=False), encoding="utf-8")
assert all((out / ("synthetic_report." + suffix)).stat().st_size > 0 for suffix in ("html", "pdf", "json"))
print("Report outputs:", sorted(p.name for p in out.glob("synthetic_report.*")))

Report outputs: ['synthetic_report.html', 'synthetic_report.json', 'synthetic_report.pdf']


For measured inputs, supply all model records and run conditions explicitly; do not reuse the synthetic parameters. C. minutissima lacks a complete verified-range preset; extinction and f_C are not supplied. The Streamlit wizard exposes the same mapping/profile/report path without writing a JSON config by hand. Hosting and observed novice usability are separate acceptance checks.